# Household Energy Usage Prediction
**Forecasting 5** | ICT30016 ICT Innovation Project | Client: ARC Research Hub
for Integrated Energy Storage Solutions

One notebook, one upload. It writes the project modules to disk, rebuilds the
daily table from the raw UCI file, runs the test suite, trains and compares the
models, rolls the forecasts up to months, validates across rolling origins, and
explains the model with SHAP.

**To run:** Runtime -> Run all, then upload
`individualhouseholdelectricpowerconsumption.zip` when the upload button appears
in step 2. Everything after that is automatic. Takes about five minutes, most of
it parsing the two million raw rows.

## 1. Setup

In [ ]:
!pip install -q xgboost shap pytest

In [ ]:
import os, json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

SEED = 42
np.random.seed(SEED)

for folder in ("src", "tests", "data", "results", "raw"):
    os.makedirs(folder, exist_ok=True)

print("folders ready")

## 2. Upload the raw data

The raw file is about 127 MB, which is too large for GitHub, so it is not in the
repository. Download it from the UCI Machine Learning Repository and upload the
zip here:

https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption

In [ ]:
from google.colab import files
up = files.upload()

with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall("raw")

RAW = "raw/household_power_consumption.txt"
print(f"{os.path.getsize(RAW) / 1e6:.1f} MB extracted")

In [ ]:
# The format matters: semicolon separated, ? for missing, dd/mm/yyyy dates.
!head -3 raw/household_power_consumption.txt

## 3. Project modules

These cells write the project's Python files to disk. They are the same files
that live in `src/` and `tests/` in the repository, so the notebook and the
repository cannot drift apart - and you can download them from the file browser
on the left if you need them.

In [ ]:
%%writefile src/build_daily.py
"""
Build the daily table from the raw UCI household power file.

Usage:  python build_daily.py --raw raw/household_power_consumption.txt

Writes data/household_daily.csv, data/manifest.json, data/data_dictionary.md
"""
import argparse, hashlib, json
from pathlib import Path

import pandas as pd

MIN_COVERAGE = 0.90          # keep a day only if this share of its minutes exists
MINUTES_PER_DAY = 1440

NUMERIC = ["Global_active_power", "Global_reactive_power", "Voltage",
           "Global_intensity", "Sub_metering_1", "Sub_metering_2",
           "Sub_metering_3"]


def load_raw(path):
    # ; separated, ? for missing, Date and Time in separate columns.
    # Combining them by hand keeps this working on any pandas version.
    df = pd.read_csv(path, sep=";", na_values=["?"], low_memory=False, dtype=str)
    df["timestamp"] = pd.to_datetime(df["Date"] + " " + df["Time"],
                                     format="%d/%m/%Y %H:%M:%S")
    for c in NUMERIC:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return (df.drop(columns=["Date", "Time"])
              .sort_values("timestamp").reset_index(drop=True))


def to_daily(df):
    g = df.set_index("timestamp").resample("D")

    out = pd.DataFrame({
        # Global_active_power is kW sampled each minute, so a day's sum is
        # kW-minutes. Divide by 60 for kWh.
        "kwh":            g["Global_active_power"].sum(min_count=1) / 60.0,
        "reactive_kvarh": g["Global_reactive_power"].sum(min_count=1) / 60.0,
        "voltage":        g["Voltage"].mean(),
        "intensity":      g["Global_intensity"].mean(),
        "sub1_wh":        g["Sub_metering_1"].sum(min_count=1),
        "sub2_wh":        g["Sub_metering_2"].sum(min_count=1),
        "sub3_wh":        g["Sub_metering_3"].sum(min_count=1),
        "minutes_seen":   g["Global_active_power"].count(),
    })
    out["coverage"] = out["minutes_seen"] / MINUTES_PER_DAY
    # active energy the three sub-meters do not account for
    out["other_wh"] = (out["kwh"] * 1000.0
                       - out["sub1_wh"] - out["sub2_wh"] - out["sub3_wh"])

    kept    = out[out["coverage"] >= MIN_COVERAGE].copy()
    dropped = out[out["coverage"] <  MIN_COVERAGE]

    kept = kept.drop(columns=["minutes_seen", "coverage"])
    kept.index.name = "date"
    return kept.reset_index(), dropped


def main(raw_path, outdir):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)

    print("reading raw file ...")
    df = load_raw(raw_path)
    blank = df[NUMERIC].isna().all(axis=1).sum()
    expected = int((df.timestamp.max() - df.timestamp.min()).total_seconds() // 60) + 1
    print(f"  rows               : {len(df):,}")
    print(f"  fully-blank rows   : {blank:,} ({blank/len(df)*100:.2f}%)")
    print(f"  range              : {df.timestamp.min()} -> {df.timestamp.max()}")
    print(f"  missing timestamps : {expected - len(df):,}")

    daily, dropped = to_daily(df)
    interior = int((daily["date"].diff().dt.days > 1).sum())
    print(f"  kept (>={MIN_COVERAGE:.0%})        : {len(daily):,}")
    print(f"  dropped (partial)  : {len(dropped):,}")
    print(f"  interior day gaps  : {interior}")

    daily.to_csv(outdir / "household_daily.csv", index=False)

    digest = hashlib.sha256(Path(raw_path).read_bytes()).hexdigest()
    (outdir / "manifest.json").write_text(json.dumps({
        "source": "UCI ML Repository - Individual household electric power consumption",
        "source_url": "https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption",
        "doi": "10.24432/C58K54",
        "raw_file": Path(raw_path).name,
        "raw_sha256": digest,
        "raw_rows": int(len(df)),
        "daily_rows": int(len(daily)),
        "date_from": str(daily.date.min().date()),
        "date_to": str(daily.date.max().date()),
        "min_coverage_rule": MIN_COVERAGE,
        "days_dropped_partial": int(len(dropped)),
        "interior_gaps": interior,
    }, indent=2))

    (outdir / "data_dictionary.md").write_text(f"""# Data dictionary - household_daily.csv

Source: UCI Individual household electric power consumption (Hebrail & Berard 2012),
one house in Sceaux, France, sampled every minute.
Raw SHA-256: `{digest[:16]}...`

Built by `build_daily.py`. A calendar day is kept only if at least
{MIN_COVERAGE:.0%} of its 1440 minutes were recorded; {len(dropped)} partial days were dropped.
No interpolation is applied.

| Column | Unit | Meaning |
|---|---|---|
| `date` | date | Calendar day |
| `kwh` | kWh | **Target.** Daily active energy. Minute kW readings summed, divided by 60. |
| `reactive_kvarh` | kVArh | Daily reactive energy, same conversion |
| `voltage` | V | Mean voltage across the day |
| `intensity` | A | Mean current intensity across the day |
| `sub1_wh` | Wh | Sub-meter 1: kitchen (dishwasher, oven, microwave) |
| `sub2_wh` | Wh | Sub-meter 2: laundry (washing machine, dryer, fridge, light) |
| `sub3_wh` | Wh | Sub-meter 3: water heater and air conditioner |
| `other_wh` | Wh | Active energy not captured by the three sub-meters (residual, not a metered circuit) |

Coverage: {daily.date.min().date()} to {daily.date.max().date()}, {len(daily)} days.
Note: {interior} interior gaps remain where whole days were dropped, so lag features
computed by row position can reach back more than the stated number of calendar days.
""")

    print(f"\nwrote {outdir}/household_daily.csv ({len(daily)} rows)")
    print(f"      {outdir}/manifest.json")
    print(f"      {outdir}/data_dictionary.md")
    return daily


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", required=True)
    ap.add_argument("--outdir", default="data")
    a = ap.parse_args()
    main(a.raw, a.outdir)

In [ ]:
%%writefile src/forecast_pipeline.py
"""
Features, chronological split, baseline + ML models, comparison table.

Usage:  python forecast_pipeline.py --data data/household_daily.csv

Writes results/comparison_table.csv, results/xgb_feature_importance.csv,
results/run_metadata.json
"""
import argparse, hashlib, json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

SEED = 42
TARGET = "kwh"

# Known only at the END of a day, so they may appear as lags but never raw.
SAME_DAY_ONLY = ["reactive_kvarh", "voltage", "intensity",
                 "sub1_wh", "sub2_wh", "sub3_wh", "other_wh"]


def load(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)


def make_features(df):
    out = df[["date", TARGET]].copy()

    for lag in (1, 2, 3, 7, 14):
        out[f"lag_{lag}"] = df[TARGET].shift(lag)
    for w in (7, 14, 28):
        out[f"roll_mean_{w}"] = df[TARGET].shift(1).rolling(w).mean()
        out[f"roll_std_{w}"]  = df[TARGET].shift(1).rolling(w).std()

    for c in [c for c in SAME_DAY_ONLY if c in df.columns]:
        out[f"{c}_lag1"] = df[c].shift(1)
        out[f"{c}_lag7"] = df[c].shift(7)

    d = out["date"].dt
    out["dow"]        = d.dayofweek
    out["month"]      = d.month
    out["doy"]        = d.dayofyear
    out["is_weekend"] = (d.dayofweek >= 5).astype(int)
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)
    out["doy_sin"] = np.sin(2 * np.pi * out["doy"] / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * out["doy"] / 365.25)

    return out.dropna().reset_index(drop=True)


def assert_no_leakage(raw, feat):
    prev = raw[["date", TARGET]].copy()
    prev["date"] = prev["date"] + pd.Timedelta(days=1)
    chk = feat[["date", "lag_1"]].merge(prev, on="date", how="inner")
    worst = (chk["lag_1"] - chk[TARGET]).abs().max()
    assert worst < 1e-9, f"lag_1 does not match the previous day (max diff {worst})"
    assert not set(SAME_DAY_ONLY) & set(feat.columns), \
        "a same-day column reached the feature table"
    print("  leakage checks passed")


def chrono_split(feat, test_frac=0.20, val_frac=0.20):
    n = len(feat)
    n_test = int(round(n * test_frac))
    trval, test = feat.iloc[: n - n_test], feat.iloc[n - n_test:]
    n_val = int(round(len(trval) * val_frac))
    return (trval.iloc[: len(trval) - n_val],
            trval.iloc[len(trval) - n_val:],
            test)


def mape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    m = y != 0                     # MAPE is undefined at zero
    return np.mean(np.abs((y[m] - p[m]) / y[m])) * 100


def score(name, y, p):
    return {"model": name,
            "MAE":  mean_absolute_error(y, p),
            "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "MAPE_%": mape(y, p)}


def main(path, outdir):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)

    raw = load(path)
    print(f"loaded {len(raw)} days  {raw.date.min().date()} -> {raw.date.max().date()}")

    feat = make_features(raw)
    assert_no_leakage(raw, feat)
    print(f"  feature table: {len(feat)} rows x {feat.shape[1]-2} features")

    train, val, test = chrono_split(feat)
    print(f"  train {len(train)} | val {len(val)} | test {len(test)}"
          f"  (test starts {test.date.min().date()})")

    drop = ["date", TARGET]
    Xtr, ytr = train.drop(columns=drop), train[TARGET]
    Xva, yva = val.drop(columns=drop),   val[TARGET]
    Xte, yte = test.drop(columns=drop),  test[TARGET]
    Xfull, yfull = pd.concat([Xtr, Xva]), pd.concat([ytr, yva])

    rows = [score("Seasonal naive (lag 7)", yte, test["lag_7"])]

    rows.append(score("Ridge regression", yte,
                      Ridge(alpha=1.0).fit(Xfull, yfull).predict(Xte)))

    rf = RandomForestRegressor(n_estimators=400, min_samples_leaf=2,
                               random_state=SEED, n_jobs=1).fit(Xfull, yfull)
    rows.append(score("Random Forest", yte, rf.predict(Xte)))

    # pick n_estimators on the validation fold, then refit on train+val
    probe = XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=4,
                         subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                         random_state=SEED, n_jobs=1,
                         early_stopping_rounds=50, eval_metric="mae")
    probe.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    best_n = probe.best_iteration + 1

    xgb = XGBRegressor(n_estimators=best_n, learning_rate=0.05, max_depth=4,
                       subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                       random_state=SEED, n_jobs=1).fit(Xfull, yfull)
    rows.append(score(f"XGBoost (n={best_n})", yte, xgb.predict(Xte)))

    res  = pd.DataFrame(rows).round(3)
    base = res.loc[0, "MAE"]
    res["MAE_vs_baseline_%"] = ((base - res["MAE"]) / base * 100).round(1)
    res = res.sort_values("MAE").reset_index(drop=True)

    print("\n" + "=" * 68)
    print(f"COMPARISON  (test = last {len(test)} days, target = daily kWh)")
    print("=" * 68)
    print(res.to_string(index=False))

    res.to_csv(outdir / "comparison_table.csv", index=False)
    (pd.Series(xgb.feature_importances_, index=Xfull.columns)
       .sort_values(ascending=False).head(15).round(4)
       .to_csv(outdir / "xgb_feature_importance.csv", header=["importance"]))

    (outdir / "run_metadata.json").write_text(json.dumps({
        "data_file": Path(path).name,
        "data_sha256_12": hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12],
        "seed": SEED,
        "rows_after_features": len(feat),
        "train": len(train), "val": len(val), "test": len(test),
        "test_start": str(test.date.min().date()),
        "xgb_best_n_estimators": int(best_n),
    }, indent=2))

    print(f"\nwrote -> {outdir}/comparison_table.csv, "
          f"xgb_feature_importance.csv, run_metadata.json")
    return res


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--outdir", default="results")
    a = ap.parse_args()
    main(a.data, a.outdir)

In [ ]:
%%writefile src/monthly_report.py
"""
Roll the daily forecasts up to a monthly view - the level the client reports at.

The model is trained on days because there are ~1,400 of them and only 47 months.
The client plans at month scale. This script closes that gap: it trains the same
XGBoost used in forecast_pipeline.py, then aggregates the test-set predictions
and actuals by calendar month and scores them there.

Daily errors partly cancel when summed over a month - overshooting Tuesday and
undershooting Wednesday nets out - so monthly percentage error should come out
below daily percentage error. That is the argument for the whole design, so the
script prints both side by side rather than only the monthly figure.

Usage:  python monthly_report.py --data data/household_daily.csv

Writes results/monthly_report.csv, results/monthly_vs_daily.csv,
       results/monthly_plot.png
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

from forecast_pipeline import (SEED, TARGET, load, make_features,
                               assert_no_leakage, chrono_split, mape)

# A month needs at least this many days present to be scored. The first and last
# months of the test window are usually partial, and a month holding four days
# is not a monthly total.
MIN_DAYS_IN_MONTH = 28


def fit_xgb(Xtr, ytr, Xva, yva):
    """Same two-stage fit as forecast_pipeline: pick n_estimators on the
    validation fold, then refit on train+val."""
    probe = XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=4,
                         subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                         random_state=SEED, n_jobs=1,
                         early_stopping_rounds=50, eval_metric="mae")
    probe.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    best_n = probe.best_iteration + 1

    model = XGBRegressor(n_estimators=best_n, learning_rate=0.05, max_depth=4,
                         subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                         random_state=SEED, n_jobs=1)
    model.fit(pd.concat([Xtr, Xva]), pd.concat([ytr, yva]))
    return model, best_n


def to_monthly(dates, actual, predicted):
    df = pd.DataFrame({"date": pd.to_datetime(dates),
                       "actual": np.asarray(actual, float),
                       "predicted": np.asarray(predicted, float)})
    df["month"] = df["date"].dt.to_period("M")

    out = (df.groupby("month")
             .agg(days=("actual", "size"),
                  actual_kwh=("actual", "sum"),
                  predicted_kwh=("predicted", "sum"))
             .reset_index())

    out["error_kwh"] = out["predicted_kwh"] - out["actual_kwh"]
    out["abs_error_kwh"] = out["error_kwh"].abs()
    out["error_pct"] = out["error_kwh"] / out["actual_kwh"] * 100
    out["complete"] = out["days"] >= MIN_DAYS_IN_MONTH
    out["month"] = out["month"].astype(str)
    return out


def plot(monthly, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    full = monthly[monthly["complete"]]
    if full.empty:
        print("  (no complete months, skipping plot)")
        return

    x = np.arange(len(full))
    fig, ax = plt.subplots(figsize=(max(6, len(full) * 0.9), 4))
    ax.bar(x - 0.2, full["actual_kwh"],    width=0.4, label="actual")
    ax.bar(x + 0.2, full["predicted_kwh"], width=0.4, label="predicted")
    ax.set_xticks(x)
    ax.set_xticklabels(full["month"], rotation=45, ha="right")
    ax.set_ylabel("kWh per month")
    ax.set_title("Monthly consumption: actual vs predicted (test period)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def main(path, outdir):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)

    raw = load(path)
    feat = make_features(raw)
    assert_no_leakage(raw, feat)

    train, val, test = chrono_split(feat)
    drop = ["date", TARGET]
    model, best_n = fit_xgb(train.drop(columns=drop), train[TARGET],
                            val.drop(columns=drop),   val[TARGET])

    pred = model.predict(test.drop(columns=drop))
    yte = test[TARGET].to_numpy()

    monthly = to_monthly(test["date"], yte, pred)
    full = monthly[monthly["complete"]]

    # Daily figures on the same test set, for the comparison that matters.
    daily_mae = float(np.mean(np.abs(pred - yte)))
    daily_mape = mape(yte, pred)

    if full.empty:
        raise SystemExit(
            f"No month in the test window has {MIN_DAYS_IN_MONTH}+ days. "
            "The test set is too short to report monthly - widen test_frac.")

    monthly_mae = float(full["abs_error_kwh"].mean())
    monthly_mape = float(full["error_pct"].abs().mean())

    compare = pd.DataFrame([
        {"level": "daily",   "unit": "kWh/day",
         "MAE": round(daily_mae, 3),   "MAPE_%": round(daily_mape, 2),
         "n": len(test)},
        {"level": "monthly", "unit": "kWh/month",
         "MAE": round(monthly_mae, 3), "MAPE_%": round(monthly_mape, 2),
         "n": int(len(full))},
    ])

    print(f"\nXGBoost (n={best_n}) | test = {len(test)} days "
          f"from {test.date.min().date()}")
    print("\nMonthly totals")
    print(monthly.round(2).to_string(index=False))
    if (~monthly["complete"]).any():
        dropped = monthly.loc[~monthly["complete"], "month"].tolist()
        print(f"\nPartial months excluded from the scores: {', '.join(dropped)}")

    print("\n" + "=" * 52)
    print("DAILY vs MONTHLY on the same predictions")
    print("=" * 52)
    print(compare.to_string(index=False))

    gap = daily_mape - monthly_mape
    if gap > 0:
        print(f"\nMonthly MAPE is {gap:.2f} points lower than daily. Day-level "
              f"errors partly cancel when summed over a month, which is why the\n"
              f"model is fitted on days and reported on months.")
    else:
        print(f"\nMonthly MAPE is NOT lower than daily ({monthly_mape:.2f} vs "
              f"{daily_mape:.2f}). Worth reporting honestly - it means the\n"
              f"daily errors are biased in one direction rather than cancelling.")

    monthly.to_csv(outdir / "monthly_report.csv", index=False)
    compare.to_csv(outdir / "monthly_vs_daily.csv", index=False)
    plot(monthly, outdir / "monthly_plot.png")

    print(f"\nwrote -> {outdir}/monthly_report.csv, monthly_vs_daily.csv, "
          f"monthly_plot.png")
    return monthly, compare


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--outdir", default="results")
    a = ap.parse_args()
    main(a.data, a.outdir)

In [ ]:
%%writefile src/rolling_validation.py
"""
Rolling-origin (forward-chaining) validation.

forecast_pipeline.py scores every model on ONE test window - the last 20% of
days. That gives a single number with no sense of how much it would move if the
window had fallen elsewhere. If that window happens to be a mild spring, the
model looks better than it is.

This script walks the origin forward instead. Fold 1 trains on the earliest
block and tests on the next; fold 2 trains on everything fold 1 saw plus its
test block, and tests on the block after that; and so on. Training data only
ever precedes test data, so no future information leaks backwards.

Each model then has K scores instead of one, and the spread across folds is the
honest statement of reliability.

Usage:  python rolling_validation.py --data data/household_daily.csv
        python rolling_validation.py --data data/household_daily.csv --folds 6

Writes results/rolling_folds.csv      (every model on every fold)
       results/rolling_summary.csv    (mean, std, worst, win rate)
       results/rolling_plot.png
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

from forecast_pipeline import (SEED, TARGET, load, make_features,
                               assert_no_leakage, mape)

BASELINE = "Seasonal naive (lag 7)"


def folds(n, k, min_train_frac=0.40):
    """Expanding-window fold boundaries.

    The first fold must already have a usable amount of history, so training
    starts at min_train_frac of the data and the remainder is divided into k
    equal test blocks.

    Yields (train_end, test_end) index pairs; train is [0, train_end).
    """
    start = int(round(n * min_train_frac))
    block = (n - start) // k
    if block < 20:
        raise SystemExit(
            f"{k} folds leaves only {block} days per test block. "
            f"Use fewer folds, or a longer series.")
    for i in range(k):
        train_end = start + i * block
        test_end = train_end + block if i < k - 1 else n
        yield train_end, test_end


def fit_models(Xtr, ytr):
    """Train the three learned models on one fold's training block.

    XGBoost needs a validation slice to stop on. It is taken as the LAST 15% of
    the training block, not a random sample, so it stays after everything the
    model trains on and before everything it is tested on.
    """
    cut = int(len(Xtr) * 0.85)
    Xf, yf = Xtr.iloc[:cut], ytr.iloc[:cut]
    Xv, yv = Xtr.iloc[cut:], ytr.iloc[cut:]

    probe = XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=4,
                         subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                         random_state=SEED, n_jobs=1,
                         early_stopping_rounds=50, eval_metric="mae")
    probe.fit(Xf, yf, eval_set=[(Xv, yv)], verbose=False)
    best_n = probe.best_iteration + 1

    return {
        "Ridge regression": Ridge(alpha=1.0).fit(Xtr, ytr),
        "Random Forest": RandomForestRegressor(
            n_estimators=400, min_samples_leaf=2,
            random_state=SEED, n_jobs=1).fit(Xtr, ytr),
        "XGBoost": XGBRegressor(
            n_estimators=best_n, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=SEED, n_jobs=1).fit(Xtr, ytr),
    }, best_n


def main(path, outdir, k):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)

    raw = load(path)
    feat = make_features(raw)
    assert_no_leakage(raw, feat)
    n = len(feat)
    print(f"{n} rows after features | {k} expanding-window folds\n")

    drop = ["date", TARGET]
    rows = []

    for i, (train_end, test_end) in enumerate(folds(n, k), start=1):
        tr = feat.iloc[:train_end]
        te = feat.iloc[train_end:test_end]

        Xtr, ytr = tr.drop(columns=drop), tr[TARGET]
        Xte, yte = te.drop(columns=drop), te[TARGET]

        preds = {BASELINE: te["lag_7"].to_numpy()}
        models, best_n = fit_models(Xtr, ytr)
        for name, m in models.items():
            preds[name] = m.predict(Xte)

        print(f"fold {i}: train {len(tr):>4} days -> test {len(te):>3} days "
              f"({te.date.min().date()} to {te.date.max().date()}), "
              f"xgb n={best_n}")

        for name, p in preds.items():
            rows.append({
                "fold": i,
                "model": name,
                "train_days": len(tr),
                "test_days": len(te),
                "test_from": str(te.date.min().date()),
                "MAE": float(np.mean(np.abs(p - yte.to_numpy()))),
                "MAPE_%": mape(yte, p),
            })

    per_fold = pd.DataFrame(rows)

    # Improvement over the baseline, computed WITHIN each fold - a baseline that
    # happens to be easy in one window must not flatter another window's model.
    base = (per_fold[per_fold.model == BASELINE]
            .set_index("fold")["MAE"].rename("base_MAE"))
    per_fold = per_fold.join(base, on="fold")
    per_fold["vs_baseline_%"] = ((per_fold["base_MAE"] - per_fold["MAE"])
                                 / per_fold["base_MAE"] * 100)

    # Win rate: in how many folds did this model beat the baseline?
    wins = (per_fold.assign(won=per_fold["MAE"] < per_fold["base_MAE"])
                    .groupby("model")["won"].mean() * 100)

    summary = (per_fold.groupby("model")
               .agg(mean_MAE=("MAE", "mean"),
                    std_MAE=("MAE", "std"),
                    worst_MAE=("MAE", "max"),
                    mean_MAPE=("MAPE_%", "mean"),
                    mean_vs_baseline=("vs_baseline_%", "mean"))
               .join(wins.rename("beat_baseline_%"))
               .sort_values("mean_MAE")
               .round(3)
               .reset_index())

    print("\n" + "=" * 78)
    print(f"PER FOLD (MAE, kWh/day)")
    print("=" * 78)
    print(per_fold.pivot(index="model", columns="fold", values="MAE")
                  .round(3).to_string())

    print("\n" + "=" * 78)
    print(f"ACROSS {k} FOLDS")
    print("=" * 78)
    print(summary.to_string(index=False))

    best = summary.iloc[0]
    print(f"\nLowest mean MAE: {best['model']} at {best['mean_MAE']:.3f} "
          f"+/- {best['std_MAE']:.3f} kWh, beating the baseline in "
          f"{best['beat_baseline_%']:.0f}% of folds.")
    if best["std_MAE"] > best["mean_MAE"] * 0.25:
        print("The spread is wide relative to the mean, so quote the range "
              "rather than the single figure.")

    per_fold.round(4).to_csv(outdir / "rolling_folds.csv", index=False)
    summary.to_csv(outdir / "rolling_summary.csv", index=False)

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 4))
        for name, g in per_fold.groupby("model"):
            style = dict(lw=2.2, ls="--") if name == BASELINE else dict(lw=1.6)
            ax.plot(g["fold"], g["MAE"], marker="o", label=name, **style)
        ax.set_xlabel("fold (origin moving forward in time)")
        ax.set_ylabel("MAE (kWh/day)")
        ax.set_title("Rolling-origin validation")
        ax.set_xticks(range(1, k + 1))
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(outdir / "rolling_plot.png", dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"(plot skipped: {e})")

    print(f"\nwrote -> {outdir}/rolling_folds.csv, rolling_summary.csv, "
          f"rolling_plot.png")
    return per_fold, summary


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--outdir", default="results")
    ap.add_argument("--folds", type=int, default=5)
    a = ap.parse_args()
    main(a.data, a.outdir, a.folds)

In [ ]:
%%writefile src/explain.py
"""
Explainability for the tree model, using TreeSHAP.

Gain importance (what forecast_pipeline.py already writes) ranks features but
says nothing about direction or size. It tells the client that other_wh_lag1
matters; it does not tell them whether a high value pushes consumption up or
down, or by how many kWh.

SHAP values do. Each one is a contribution in kWh to a single day's
prediction, and they sum to the difference between that prediction and the
average prediction. That makes them reportable in the client's own units
rather than as a unitless score.

Usage:  python explain.py --data data/household_daily.csv
        python explain.py --data data/household_daily.csv --model rf

Writes results/shap_summary.csv       mean |SHAP| and direction per feature
       results/shap_drivers.md        the same thing in plain English
       results/shap_beeswarm.png      distribution of effects
       results/shap_waterfall_*.png   two single days explained end to end
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from forecast_pipeline import (SEED, TARGET, load, make_features,
                               assert_no_leakage, chrono_split)

# Readable names for the report. Anything not listed falls back to the raw
# column name, so this map can stay partial.
PRETTY = {
    "lag_1": "yesterday's consumption",
    "lag_2": "consumption two days ago",
    "lag_3": "consumption three days ago",
    "lag_7": "same day last week",
    "lag_14": "same day two weeks ago",
    "roll_mean_7": "average of the last 7 days",
    "roll_mean_14": "average of the last 14 days",
    "roll_mean_28": "average of the last 28 days",
    "roll_std_7": "volatility over the last 7 days",
    "roll_std_14": "volatility over the last 14 days",
    "roll_std_28": "volatility over the last 28 days",
    "dow": "day of week",
    "dow_sin": "day of week (cyclical)",
    "dow_cos": "day of week (cyclical)",
    "doy": "day of year",
    "doy_sin": "time of year (cyclical)",
    "doy_cos": "time of year (cyclical)",
    "month": "month",
    "is_weekend": "weekend or weekday",
    "other_wh_lag1": "yesterday's unmetered load (residual)",
    "other_wh_lag7": "unmetered load same day last week",
    "sub1_wh_lag1": "yesterday's kitchen circuit",
    "sub2_wh_lag1": "yesterday's laundry circuit",
    "sub3_wh_lag1": "yesterday's water heater and A/C",
    "sub1_wh_lag7": "kitchen circuit same day last week",
    "sub2_wh_lag7": "laundry circuit same day last week",
    "sub3_wh_lag7": "water heater and A/C same day last week",
    "voltage_lag1": "yesterday's mean voltage",
    "voltage_lag7": "mean voltage same day last week",
    "intensity_lag1": "yesterday's mean current",
    "intensity_lag7": "mean current same day last week",
    "reactive_kvarh_lag1": "yesterday's reactive energy",
    "reactive_kvarh_lag7": "reactive energy same day last week",
}


def pretty(name):
    return PRETTY.get(name, name.replace("_", " "))


def build_model(kind, Xtr, ytr, Xva, yva):
    if kind == "rf":
        from sklearn.ensemble import RandomForestRegressor
        m = RandomForestRegressor(n_estimators=400, min_samples_leaf=2,
                                  random_state=SEED, n_jobs=1)
        m.fit(pd.concat([Xtr, Xva]), pd.concat([ytr, yva]))
        return m, "Random Forest"

    from xgboost import XGBRegressor
    probe = XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=4,
                         subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                         random_state=SEED, n_jobs=1,
                         early_stopping_rounds=50, eval_metric="mae")
    probe.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    best_n = probe.best_iteration + 1

    m = XGBRegressor(n_estimators=best_n, learning_rate=0.05, max_depth=4,
                     subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                     random_state=SEED, n_jobs=1)
    m.fit(pd.concat([Xtr, Xva]), pd.concat([ytr, yva]))
    return m, f"XGBoost (n={best_n})"


def summarise(shap_values, X):
    """Per-feature: average size of effect, and whether high feature values
    push the prediction up or down.

    Direction is the correlation between the feature's value and its own SHAP
    value. Positive means a high value raises the forecast. Near zero means
    the effect is not monotonic - worth saying rather than hiding.
    """
    rows = []
    for i, col in enumerate(X.columns):
        sv = shap_values[:, i]
        xv = X[col].to_numpy(float)

        if np.std(xv) < 1e-12 or np.std(sv) < 1e-12:
            corr = 0.0
        else:
            corr = float(np.corrcoef(xv, sv)[0, 1])

        rows.append({
            "feature": col,
            "plain_name": pretty(col),
            "mean_abs_shap_kwh": float(np.mean(np.abs(sv))),
            "max_abs_shap_kwh": float(np.max(np.abs(sv))),
            "direction_corr": corr,
            "direction": ("higher raises the forecast" if corr > 0.2
                          else "higher lowers the forecast" if corr < -0.2
                          else "mixed / not monotonic"),
        })

    return (pd.DataFrame(rows)
            .sort_values("mean_abs_shap_kwh", ascending=False)
            .reset_index(drop=True))


def write_drivers(summary, model_name, base_value, n_rows, path, top=6):
    lines = [
        "# What drives the daily forecast",
        "",
        f"Model: {model_name}. Explained on {n_rows} held-out test days using "
        f"TreeSHAP.",
        "",
        f"The model's average prediction across these days is "
        f"**{base_value:.2f} kWh**. Each figure below is how far that one "
        f"feature moves a day's forecast away from that average, in kWh, "
        f"averaged over all the test days.",
        "",
    ]
    for i, r in summary.head(top).iterrows():
        lines.append(
            f"{i + 1}. **{r['plain_name']}** - moves the forecast by "
            f"{r['mean_abs_shap_kwh']:.2f} kWh on an average day, up to "
            f"{r['max_abs_shap_kwh']:.2f} kWh on the most affected day. "
            f"{r['direction'].capitalize()}."
        )

    lines += [
        "",
        "## Reading this",
        "",
        "SHAP values are additive: for any single day, the base value plus "
        "every feature's contribution equals that day's prediction exactly. "
        "So these numbers can be checked, not just trusted.",
        "",
        "A feature marked *mixed / not monotonic* matters, but not in one "
        "direction - typically because its effect depends on another feature. "
        "Calendar features behave this way by nature.",
        "",
    ]

    if not summary.empty and "other_wh" in summary.iloc[0]["feature"]:
        lines += [
            "## Caveat on the top driver",
            "",
            "`other_wh` is a **residual**: total active energy minus the three "
            "sub-metered circuits. It is not a measured appliance group. Its "
            "importance says that load outside the three monitored circuits "
            "carries much of the day-to-day variation, which is a finding "
            "about the metering coverage of this house, not about a specific "
            "appliance.",
            "",
        ]

    Path(path).write_text("\n".join(lines))


def main(path, outdir, kind, sample):
    try:
        import shap
    except ImportError:
        raise SystemExit("shap is not installed. Run:  pip install shap")

    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)

    raw = load(path)
    feat = make_features(raw)
    assert_no_leakage(raw, feat)
    train, val, test = chrono_split(feat)

    drop = ["date", TARGET]
    model, model_name = build_model(
        kind, train.drop(columns=drop), train[TARGET],
        val.drop(columns=drop), val[TARGET])

    Xte = test.drop(columns=drop)
    if sample and len(Xte) > sample:
        Xte = Xte.iloc[:sample]
    print(f"{model_name} | explaining {len(Xte)} test days")

    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(Xte)
    base = float(np.ravel(explainer.expected_value)[0])

    # Additivity check. If this fails the explanation is not faithful to the
    # model and should not be reported.
    recon = base + sv.sum(axis=1)
    worst = float(np.abs(recon - model.predict(Xte)).max())
    assert worst < 1e-2, f"SHAP values do not reconstruct predictions ({worst})"
    print(f"  additivity check passed (max deviation {worst:.2e} kWh)")

    summary = summarise(sv, Xte)
    summary.round(4).to_csv(outdir / "shap_summary.csv", index=False)
    write_drivers(summary, model_name, base, len(Xte),
                  outdir / "shap_drivers.md")

    print(f"\n  base value: {base:.2f} kWh")
    print("\n  top drivers (mean |SHAP|, kWh):")
    for _, r in summary.head(8).iterrows():
        print(f"    {r['mean_abs_shap_kwh']:6.3f}  {r['plain_name']:<42}"
              f" {r['direction']}")

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        shap.summary_plot(sv, Xte, show=False, max_display=15)
        plt.tight_layout()
        plt.savefig(outdir / "shap_beeswarm.png", dpi=150)
        plt.close()

        # Two individual days: the highest and lowest actual consumption in
        # the explained window. These are the slides that land in a demo.
        y = test[TARGET].iloc[:len(Xte)]
        for label, idx in (("highest_day", int(np.argmax(y.to_numpy()))),
                           ("lowest_day", int(np.argmin(y.to_numpy())))):
            ex = shap.Explanation(values=sv[idx], base_values=base,
                                  data=Xte.iloc[idx],
                                  feature_names=[pretty(c) for c in Xte.columns])
            shap.plots.waterfall(ex, max_display=12, show=False)
            plt.tight_layout()
            plt.savefig(outdir / f"shap_waterfall_{label}.png", dpi=150)
            plt.close()
        print("\n  wrote beeswarm and two waterfall plots")
    except Exception as e:
        print(f"\n  (plots skipped: {e})")

    print(f"\nwrote -> {outdir}/shap_summary.csv, shap_drivers.md, *.png")
    return summary


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--outdir", default="results")
    ap.add_argument("--model", default="xgb", choices=["xgb", "rf"])
    ap.add_argument("--sample", type=int, default=0,
                    help="explain only the first N test days (0 = all)")
    a = ap.parse_args()
    main(a.data, a.outdir, a.model, a.sample)

## 4. Test suite

Thirty-two tests covering the Week 8 QA plan: leakage (including tamper tests
that deliberately break the rule and confirm the assertion fires), the
kW-minute to kWh conversion, the 90% coverage rule, chronological splitting,
MAPE's behaviour at zero, the monthly rollup arithmetic, and the rolling-origin
fold boundaries.

In [ ]:
%%writefile tests/conftest.py
"""
Shared fixtures. Lets the tests import from src/ without installing anything.
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

# Where the real generated files live, if they have been built on this machine.
DATA = ROOT / "data" / "household_daily.csv"
MANIFEST = ROOT / "data" / "manifest.json"


@pytest.fixture
def minute_frame():
    """Two days of minute-level readings in the raw file's shape.

    Day 1 is complete (1440 minutes). Day 2 has only 100 minutes, so the
    coverage rule must drop it. Global_active_power is a constant 1.2 kW,
    which makes the kWh conversion checkable by hand:
    1440 minutes x 1.2 kW / 60 = 28.8 kWh.
    """
    full = pd.date_range("2007-01-01 00:00:00", periods=1440, freq="min")
    part = pd.date_range("2007-01-02 00:00:00", periods=100, freq="min")
    ts = full.append(part)

    return pd.DataFrame({
        "timestamp": ts,
        "Global_active_power": 1.2,
        "Global_reactive_power": 0.1,
        "Voltage": 240.0,
        "Global_intensity": 5.0,
        "Sub_metering_1": 1.0,
        "Sub_metering_2": 2.0,
        "Sub_metering_3": 3.0,
    })


@pytest.fixture
def daily_frame():
    """A daily table with the real schema, long enough to build features from.

    Values are synthetic. Nothing here is a project result - these tests check
    that the code behaves correctly, not what the household consumed.
    """
    rng = np.random.default_rng(0)
    n = 400
    date = pd.date_range("2007-01-01", periods=n, freq="D")
    kwh = 26 + 6 * np.sin(2 * np.pi * date.dayofyear / 365.25) \
            + rng.normal(0, 3, n)
    kwh = np.clip(kwh, 1, None)

    df = pd.DataFrame({
        "date": date,
        "kwh": kwh,
        "reactive_kvarh": kwh * 0.08,
        "voltage": 240 + rng.normal(0, 2, n),
        "intensity": kwh / 5,
        "sub1_wh": rng.uniform(0, 2000, n),
        "sub2_wh": rng.uniform(0, 3000, n),
        "sub3_wh": rng.uniform(0, 9000, n),
    })
    df["other_wh"] = df.kwh * 1000 - df.sub1_wh - df.sub2_wh - df.sub3_wh
    return df


needs_real_data = pytest.mark.skipif(
    not DATA.exists(),
    reason="data/household_daily.csv not built on this machine - "
           "run src/build_daily.py first")

In [ ]:
%%writefile tests/test_data_pipeline.py
"""
Unit and regression tests for the raw-to-daily stage (feature F6 in the QA plan).

Objective: "Confirm the daily table can be rebuilt from raw without drift."
"""
import json

import pandas as pd
import pytest

import build_daily
from conftest import DATA, MANIFEST, needs_real_data


# --------------------------------------------------------------- unit tests

def test_kwh_conversion_divides_by_sixty(minute_frame):
    """Global_active_power is kW sampled per minute, so a day's energy is the
    sum of the readings divided by 60. Getting this wrong inflates daily
    consumption roughly 60-fold, which is the worst bug this project could
    ship, so it is checked against a hand-computed value."""
    kept, _ = build_daily.to_daily(minute_frame)

    day1 = kept.loc[kept.date == pd.Timestamp("2007-01-01")].iloc[0]
    assert day1["kwh"] == pytest.approx(1440 * 1.2 / 60)   # 28.8
    assert day1["kwh"] == pytest.approx(28.8)


def test_partial_day_is_dropped(minute_frame):
    """A day with 100 of 1440 minutes looks like a very low consumption day.
    It must be dropped, not kept and not interpolated."""
    kept, dropped = build_daily.to_daily(minute_frame)

    assert len(kept) == 1
    assert kept.date.iloc[0] == pd.Timestamp("2007-01-01")
    assert pd.Timestamp("2007-01-02") in dropped.index


def test_coverage_threshold_is_ninety_percent():
    """The rule is documented in the data dictionary and the report, so it is
    pinned here. Changing it should break a test, not slip through."""
    assert build_daily.MIN_COVERAGE == 0.90
    assert build_daily.MINUTES_PER_DAY == 1440


def test_day_just_above_threshold_is_kept():
    """1296 minutes is exactly 90%, so it must survive."""
    ts = pd.date_range("2007-03-01 00:00:00", periods=1296, freq="min")
    df = pd.DataFrame({"timestamp": ts, "Global_active_power": 1.0,
                       "Global_reactive_power": 0.0, "Voltage": 240.0,
                       "Global_intensity": 4.0, "Sub_metering_1": 0.0,
                       "Sub_metering_2": 0.0, "Sub_metering_3": 0.0})
    kept, dropped = build_daily.to_daily(df)
    assert len(kept) == 1 and len(dropped) == 0


def test_other_wh_is_a_residual(minute_frame):
    """other_wh is total active energy minus the three sub-meters. It is a
    residual, not a metered circuit, and the report says so - this checks the
    arithmetic matches that description."""
    kept, _ = build_daily.to_daily(minute_frame)
    r = kept.iloc[0]
    assert r["other_wh"] == pytest.approx(
        r["kwh"] * 1000 - r["sub1_wh"] - r["sub2_wh"] - r["sub3_wh"])


def test_helper_columns_do_not_reach_the_output(minute_frame):
    """minutes_seen and coverage are working columns. If they leaked into the
    daily table they would become features, and coverage is not something a
    forecast can know in advance."""
    kept, _ = build_daily.to_daily(minute_frame)
    assert "minutes_seen" not in kept.columns
    assert "coverage" not in kept.columns


# ------------------------------------------------- regression against manifest

@needs_real_data
def test_daily_table_matches_manifest():
    """The regression test named in the Week 8 QA plan: the committed manifest
    and the built table must agree on row count and date range. If a second
    member rebuilds and this fails, the cleaning rules have drifted."""
    daily = pd.read_csv(DATA, parse_dates=["date"])
    man = json.loads(MANIFEST.read_text())

    assert len(daily) == man["daily_rows"]
    assert str(daily.date.min().date()) == man["date_from"]
    assert str(daily.date.max().date()) == man["date_to"]
    assert man["min_coverage_rule"] == build_daily.MIN_COVERAGE


@needs_real_data
def test_daily_table_is_sane():
    """Order, uniqueness, no nulls in the target, and a plausible magnitude.
    A single household averaging hundreds of kWh a day would mean the /60
    conversion had been lost again."""
    daily = pd.read_csv(DATA, parse_dates=["date"])

    assert daily.date.is_monotonic_increasing
    assert not daily.date.duplicated().any()
    assert daily.kwh.notna().all()
    assert (daily.kwh > 0).all()
    assert 5 < daily.kwh.mean() < 100, \
        f"mean of {daily.kwh.mean():.1f} kWh/day is not a plausible household"

In [ ]:
%%writefile tests/test_leakage.py
"""
Leakage tests (QA plan objective: "Confirm no leakage happens").

The pipeline already asserts this at runtime. These tests add the part a
runtime assertion cannot do: they deliberately break the rule and check the
assertion actually fires. An assertion nobody has ever seen fail is not
evidence of anything.
"""
import pandas as pd
import pytest

import forecast_pipeline as fp


def test_lag_1_equals_previous_day(daily_frame):
    feat = fp.make_features(daily_frame)
    prev = daily_frame[["date", "kwh"]].copy()
    prev["date"] = prev["date"] + pd.Timedelta(days=1)
    chk = feat[["date", "lag_1"]].merge(prev, on="date")

    assert len(chk) > 0
    assert (chk.lag_1 - chk.kwh).abs().max() < 1e-9


def test_lag_7_equals_a_week_earlier(daily_frame):
    feat = fp.make_features(daily_frame)
    prev = daily_frame[["date", "kwh"]].copy()
    prev["date"] = prev["date"] + pd.Timedelta(days=7)
    chk = feat[["date", "lag_7"]].merge(prev, on="date")

    assert (chk.lag_7 - chk.kwh).abs().max() < 1e-9


def test_same_day_columns_never_appear_raw(daily_frame):
    """Voltage, the sub-meters and the residual are only known once the day is
    over. They may appear as lags; the bare column names must not."""
    feat = fp.make_features(daily_frame)
    assert not set(fp.SAME_DAY_ONLY) & set(feat.columns)


def test_same_day_columns_are_available_as_lags(daily_frame):
    """The flip side: dropping them entirely would throw away real signal, so
    confirm the lagged versions survived."""
    feat = fp.make_features(daily_frame)
    for c in fp.SAME_DAY_ONLY:
        assert f"{c}_lag1" in feat.columns
        assert f"{c}_lag7" in feat.columns


def test_rolling_statistics_exclude_today(daily_frame):
    """roll_mean_7 must be built from the seven days BEFORE the target day.
    Including today would put the answer inside the feature."""
    feat = fp.make_features(daily_frame)
    row = feat.iloc[50]
    window = daily_frame.loc[daily_frame.date < row["date"], "kwh"].tail(7)

    assert row["roll_mean_7"] == pytest.approx(window.mean())


def test_assertion_fires_when_a_same_day_column_is_smuggled_in(daily_frame):
    """Tamper test. Add voltage to the feature table unlagged and confirm
    assert_no_leakage rejects it."""
    feat = fp.make_features(daily_frame)
    feat["voltage"] = 240.0

    with pytest.raises(AssertionError):
        fp.assert_no_leakage(daily_frame, feat)


def test_assertion_fires_when_lags_are_shifted_the_wrong_way(daily_frame):
    """Tamper test. Shift the target backwards instead of forwards - the
    classic off-by-one that turns a forecast into a lookup of tomorrow."""
    feat = fp.make_features(daily_frame)
    feat["lag_1"] = daily_frame["kwh"].shift(-1).iloc[len(daily_frame) - len(feat):].values

    with pytest.raises(AssertionError):
        fp.assert_no_leakage(daily_frame, feat)


def test_clean_feature_table_passes(daily_frame):
    """And the honest case must not raise."""
    feat = fp.make_features(daily_frame)
    fp.assert_no_leakage(daily_frame, feat)      # no exception = pass


def test_no_nulls_survive_feature_building(daily_frame):
    """Lags and rolling windows create nulls at the start of the series.
    They are dropped, not filled, so nothing should remain."""
    feat = fp.make_features(daily_frame)
    assert feat.notna().all().all()
    assert len(feat) < len(daily_frame)     # the warm-up rows were removed

In [ ]:
%%writefile tests/test_evaluation.py
"""
Tests for splitting, metrics and the monthly rollup.

QA plan objectives: "Confirm fairness for all model comparison" and
"Confirm the accuracy result holds when the test window moves".
"""
import numpy as np
import pandas as pd
import pytest

import forecast_pipeline as fp
import monthly_report as mr
import rolling_validation as rv


# ------------------------------------------------------------------- splitting

def test_split_is_chronological(daily_frame):
    """Every training day must precede every validation day, and every
    validation day every test day. A shuffled split would still produce
    good-looking scores, which is exactly why this is tested."""
    feat = fp.make_features(daily_frame)
    train, val, test = fp.chrono_split(feat)

    assert train.date.max() < val.date.min()
    assert val.date.max() < test.date.min()


def test_split_loses_no_rows(daily_frame):
    feat = fp.make_features(daily_frame)
    train, val, test = fp.chrono_split(feat)
    assert len(train) + len(val) + len(test) == len(feat)


def test_split_proportions(daily_frame):
    """20% test, then 20% of the remainder as validation."""
    feat = fp.make_features(daily_frame)
    train, val, test = fp.chrono_split(feat)
    n = len(feat)

    assert len(test) == pytest.approx(0.20 * n, abs=1)
    assert len(val) == pytest.approx(0.20 * 0.80 * n, abs=1)


# --------------------------------------------------------------------- metrics

def test_mape_ignores_zero_actuals():
    """MAPE is undefined when the actual is zero. Those rows are masked out
    rather than producing inf, which would silently poison the mean."""
    y = np.array([0.0, 10.0, 20.0])
    p = np.array([5.0, 11.0, 18.0])

    got = fp.mape(y, p)
    expected = np.mean([1 / 10, 2 / 20]) * 100      # the zero row is excluded

    assert got == pytest.approx(expected)
    assert np.isfinite(got)


def test_mape_is_zero_for_perfect_predictions():
    y = np.array([10.0, 20.0, 30.0])
    assert fp.mape(y, y) == pytest.approx(0.0)


def test_perfect_prediction_scores_zero_on_everything():
    y = np.array([10.0, 20.0, 30.0])
    s = fp.score("perfect", y, y)
    assert s["MAE"] == pytest.approx(0.0)
    assert s["RMSE"] == pytest.approx(0.0)
    assert s["MAPE_%"] == pytest.approx(0.0)


def test_rmse_penalises_large_errors_more_than_mae():
    """One big miss should move RMSE further than MAE. This is why both are
    reported rather than just one."""
    y = np.array([10.0, 10.0, 10.0, 10.0])
    p = np.array([10.0, 10.0, 10.0, 18.0])
    s = fp.score("spiky", y, p)
    assert s["RMSE"] > s["MAE"]


# -------------------------------------------------------------- monthly rollup

def test_monthly_totals_sum_the_daily_values():
    dates = pd.date_range("2010-01-01", periods=62, freq="D")
    actual = np.full(62, 10.0)
    pred = np.full(62, 11.0)

    out = mr.to_monthly(dates, actual, pred)

    jan = out[out.month == "2010-01"].iloc[0]
    assert jan["days"] == 31
    assert jan["actual_kwh"] == pytest.approx(310.0)
    assert jan["predicted_kwh"] == pytest.approx(341.0)
    assert jan["error_kwh"] == pytest.approx(31.0)
    assert jan["error_pct"] == pytest.approx(10.0)


def test_partial_months_are_flagged_incomplete():
    """A test window starting mid-month produces a stub month. It must be
    marked so it never enters the averages as if it were a real month."""
    dates = pd.date_range("2010-01-29", periods=34, freq="D")   # 3 + 28 + 3
    out = mr.to_monthly(dates, np.full(34, 10.0), np.full(34, 10.0))

    jan = out[out.month == "2010-01"].iloc[0]
    assert jan["days"] == 3
    assert not jan["complete"]

    feb = out[out.month == "2010-02"].iloc[0]
    assert feb["complete"]


def test_min_days_in_month_threshold():
    assert mr.MIN_DAYS_IN_MONTH == 28


# ------------------------------------------------------- rolling-origin folds

def test_folds_never_train_on_the_future():
    """The core property of forward chaining: each fold's training block ends
    exactly where its test block starts."""
    for train_end, test_end in rv.folds(n=1000, k=5):
        assert train_end < test_end


def test_folds_expand_and_do_not_overlap():
    bounds = list(rv.folds(n=1000, k=5))

    train_ends = [t for t, _ in bounds]
    assert train_ends == sorted(train_ends)          # training grows
    for (_, prev_test_end), (next_train_end, _) in zip(bounds, bounds[1:]):
        assert prev_test_end == next_train_end       # blocks are contiguous


def test_folds_cover_the_tail_of_the_series():
    """The last fold must reach the final row, so no data is quietly unused."""
    bounds = list(rv.folds(n=1000, k=5))
    assert bounds[-1][1] == 1000


def test_first_fold_has_enough_history():
    bounds = list(rv.folds(n=1000, k=5))
    assert bounds[0][0] >= 400          # 40% minimum training block


def test_too_many_folds_is_refused():
    """Twenty folds on a short series would leave test blocks of a few days,
    which produces meaningless scores. Better to fail loudly."""
    with pytest.raises(SystemExit):
        list(rv.folds(n=300, k=20))

In [ ]:
%%writefile tests/test_prediction.py
"""
Tests for the forecasting path - the part that produces numbers for days that
have not happened yet.

The rest of the suite checks that the models are measured honestly. These check
that the thing a client would actually run does what it claims.
"""
import numpy as np
import pandas as pd
import pytest

import predict as pr
import train_model as tm


@pytest.fixture(scope="module")
def trained(tmp_path_factory):
    """Train both models once on the synthetic daily table and reuse them."""
    import forecast_pipeline as fp

    rng = np.random.default_rng(2)
    n = 500
    date = pd.date_range("2007-01-01", periods=n, freq="D")
    kwh = np.clip(26 + 6 * np.sin(2 * np.pi * date.dayofyear / 365.25)
                  + rng.normal(0, 3, n), 1, None)
    df = pd.DataFrame({
        "date": date, "kwh": kwh, "reactive_kvarh": kwh * 0.08,
        "voltage": 240 + rng.normal(0, 2, n), "intensity": kwh / 5,
        "sub1_wh": rng.uniform(0, 2000, n),
        "sub2_wh": rng.uniform(0, 3000, n),
        "sub3_wh": rng.uniform(0, 9000, n),
    })
    df["other_wh"] = df.kwh * 1000 - df.sub1_wh - df.sub2_wh - df.sub3_wh

    d = tmp_path_factory.mktemp("trained")
    data = d / "daily.csv"
    df.to_csv(data, index=False)
    tm.main(str(data), str(d / "models"))
    return df, str(data), str(d / "models")


# ------------------------------------------------- feature building by hand

def test_lag_1_is_the_most_recent_value():
    series = [10.0, 20.0, 30.0, 40.0, 50.0] * 10
    row = pr.history_features(series, "2010-06-15")
    assert row["lag_1"] == series[-1]
    assert row["lag_7"] == series[-7]


def test_rolling_mean_uses_only_past_values():
    series = list(range(1, 41))              # 1..40
    row = pr.history_features(series, "2010-06-15")
    assert row["roll_mean_7"] == pytest.approx(np.mean(series[-7:]))
    assert row["roll_mean_28"] == pytest.approx(np.mean(series[-28:]))


def test_calendar_features_for_a_known_date():
    # 2010-06-15 was a Tuesday, day 166 of the year.
    f = pr.calendar_features("2010-06-15")
    assert f["dow"] == 1
    assert f["month"] == 6
    assert f["doy"] == 166
    assert f["is_weekend"] == 0


def test_weekend_flag():
    assert pr.calendar_features("2010-06-19")["is_weekend"] == 1   # Saturday
    assert pr.calendar_features("2010-06-20")["is_weekend"] == 1   # Sunday
    assert pr.calendar_features("2010-06-21")["is_weekend"] == 0   # Monday


def test_cyclical_encodings_stay_on_the_unit_circle():
    for date in ("2010-01-01", "2010-06-15", "2010-12-31"):
        f = pr.calendar_features(date)
        assert f["dow_sin"] ** 2 + f["dow_cos"] ** 2 == pytest.approx(1.0)
        assert f["doy_sin"] ** 2 + f["doy_cos"] ** 2 == pytest.approx(1.0)


# ------------------------------------------------------------- the forecast

def test_forecast_starts_the_day_after_the_last_observation(trained):
    df, _, models = trained
    fc, _, _ = pr.forecast(df, 7, models)

    expected = pd.Timestamp(df.date.iloc[-1]) + pd.Timedelta(days=1)
    assert fc.date.iloc[0] == expected


def test_forecast_dates_are_consecutive(trained):
    df, _, models = trained
    fc, _, _ = pr.forecast(df, 14, models)
    assert (fc.date.diff().dt.days.dropna() == 1).all()
    assert len(fc) == 14


def test_first_step_uses_the_accurate_model_and_the_rest_recursive(trained):
    """Only tomorrow can use yesterday's end-of-day readings. Everything
    further out must fall back to the reduced feature set."""
    df, _, models = trained
    fc, _, _ = pr.forecast(df, 5, models)

    assert fc.model.iloc[0] == "next_day"
    assert set(fc.model.iloc[1:]) == {"horizon"}


def test_predictions_are_physically_plausible(trained):
    df, _, models = trained
    fc, _, _ = pr.forecast(df, 30, models)

    assert (fc.predicted_kwh > 0).all()
    # Nothing should wander outside the range the house has ever recorded by
    # more than a wide margin - a runaway recursive forecast would.
    assert fc.predicted_kwh.max() < df.kwh.max() * 1.5
    assert fc.predicted_kwh.min() > df.kwh.min() * 0.5


def test_uncertainty_band_widens_with_distance(trained):
    """Day 30 must not be presented with the same confidence as day 1."""
    df, _, models = trained
    fc, _, _ = pr.forecast(df, 20, models)

    width = fc.upper_kwh - fc.lower_kwh
    assert width.iloc[-1] > width.iloc[0]
    assert fc.expected_MAE.is_monotonic_increasing


def test_band_contains_the_point_forecast(trained):
    df, _, models = trained
    fc, _, _ = pr.forecast(df, 10, models)
    assert (fc.lower_kwh <= fc.predicted_kwh).all()
    assert (fc.predicted_kwh <= fc.upper_kwh).all()


def test_lower_bound_never_goes_negative(trained):
    """A house cannot consume negative energy, so the band is clipped."""
    df, _, models = trained
    fc, _, _ = pr.forecast(df, 60, models)
    assert (fc.lower_kwh >= 0).all()


def test_forecast_is_deterministic(trained):
    """Same data, same models, same answer - the seed is fixed and nothing
    samples at prediction time."""
    df, _, models = trained
    a, _, _ = pr.forecast(df, 10, models)
    b, _, _ = pr.forecast(df, 10, models)
    pd.testing.assert_frame_equal(a, b)


def test_horizon_model_never_sees_end_of_day_features(trained):
    """The whole reason two models exist. If a sub-meter lag reached the
    horizon feature list, recursive forecasting would be impossible."""
    _, _, models = trained
    _, meta = pr.load_model(f"{models}/horizon.joblib")

    for col in ("other_wh", "sub1_wh", "sub2_wh", "sub3_wh",
                "voltage", "intensity", "reactive_kvarh"):
        assert not any(f.startswith(col) for f in meta["features"]), \
            f"{col} leaked into the horizon model"


def test_next_day_model_does_use_them(trained):
    """And the one-step model should, since they are genuinely available."""
    _, _, models = trained
    _, meta = pr.load_model(f"{models}/next_day.joblib")
    assert "other_wh_lag1" in meta["features"]


# ------------------------------------------------------------ monthly view

def test_monthly_flags_months_the_forecast_only_partly_covers(trained):
    df, _, models = trained
    fc, _, _ = pr.forecast(df, 20, models)
    monthly = pr.to_monthly(fc, df.date.iloc[-1])

    # 20 days cannot fill a calendar month, so nothing may be marked complete.
    assert not monthly.complete.any()
    assert monthly.days_forecast.sum() == 20


def test_monthly_totals_match_the_daily_sum(trained):
    df, _, models = trained
    fc, _, _ = pr.forecast(df, 45, models)
    monthly = pr.to_monthly(fc, df.date.iloc[-1])

    assert monthly.predicted_kwh.sum() == pytest.approx(fc.predicted_kwh.sum())


# --------------------------------------------------------- saved model card

def test_model_metadata_records_provenance(trained):
    _, _, models = trained
    for name in ("next_day", "horizon"):
        _, meta = pr.load_model(f"{models}/{name}.joblib")
        for key in ("features", "n_estimators", "trained_rows",
                    "trained_from", "trained_to", "data_sha256_12", "seed"):
            assert key in meta, f"{name} metadata missing {key}"
        assert meta["seed"] == 42


def test_model_card_was_written(trained):
    from pathlib import Path
    _, _, models = trained
    card = Path(models) / "model_card.md"
    assert card.exists()
    text = card.read_text()
    assert "Limits" in text and "One household" in text

In [ ]:
# Two tests need the daily table, which does not exist yet, so they skip here.
# The suite runs again at the end once the table has been built.
!python -m pytest tests/ -q

## 5. Build the daily table

`Global_active_power` is kilowatts sampled every minute, not energy. A day's
energy in kWh is the sum of the minute readings divided by 60. Without that
division daily consumption comes out around 1,571 kWh instead of about 26, so
it is the most important line in the pipeline.

A calendar day is kept only if at least 90% of its 1,440 minutes were recorded.
Partial days are dropped rather than interpolated, because a day holding three
hours of data looks like an extremely low consumption day and would corrupt both
the target and every lag built from it.

In [ ]:
!python src/build_daily.py --raw raw/household_power_consumption.txt

In [ ]:
daily = pd.read_csv("data/household_daily.csv", parse_dates=["date"])
manifest = json.load(open("data/manifest.json"))

print(f"rows      {len(daily)}")
print(f"range     {daily.date.min().date()} -> {daily.date.max().date()}")
print(f"mean kwh  {daily.kwh.mean():.2f}")
print(f"min/max   {daily.kwh.min():.2f} / {daily.kwh.max():.2f}")
print(f"raw sha   {manifest['raw_sha256'][:16]}...")

daily.head()

In [ ]:
daily.plot(x="date", y="kwh", figsize=(13, 3), lw=0.6, legend=False)
plt.title("Daily household consumption (kWh)")
plt.ylabel("kWh")
plt.tight_layout()
plt.savefig("results/daily_series.png", dpi=150)
plt.show()

In [ ]:
# Days dropped by the coverage rule leave gaps, so lag features computed by row
# position can reach back further than the stated number of calendar days.
gap = daily.date.diff().dt.days
breaks = gap[gap > 1]
print(f"{len(breaks)} interior gaps after the coverage filter")
if len(breaks):
    display(pd.DataFrame({
        "gap_before": daily.date[breaks.index - 1].values,
        "resumes":    daily.date[breaks.index].values,
        "days_missing": breaks.values - 1,
    }))

## 6. Features, split and model comparison

Seven columns in the daily table are only known once the day is over: reactive
energy, voltage, current, the three sub-meters and the residual. Using any of
them to predict the same day would be using the answer to compute the answer, so
they appear only as lags, and assertions enforce that on every run.

The split is chronological - train, then validation, then test, in time order.
Random k-fold cross-validation is invalid for dependent data (Bergmeir and
Benitez 2012).

In [ ]:
!python src/forecast_pipeline.py --data data/household_daily.csv

In [ ]:
comparison = pd.read_csv("results/comparison_table.csv")
display(comparison)

imp = pd.read_csv("results/xgb_feature_importance.csv",
                  index_col=0).squeeze("columns")
imp.sort_values().tail(12).plot.barh(figsize=(6, 4))
plt.title("XGBoost feature importance (gain)")
plt.tight_layout()
plt.savefig("results/feature_importance.png", dpi=150)
plt.show()

## 7. Monthly reporting rollup

The model is fitted on days because there are roughly 1,400 of them and only 47
months. The client plans at month scale. This step closes that gap.

Daily errors partly cancel when summed over a month - overshooting Tuesday and
undershooting Wednesday nets out - so monthly percentage error should land below
daily percentage error. That is the argument for the whole design, so both are
printed side by side rather than just the monthly figure.

In [ ]:
!python src/monthly_report.py --data data/household_daily.csv

In [ ]:
display(pd.read_csv("results/monthly_vs_daily.csv"))
display(pd.read_csv("results/monthly_report.csv"))
Image("results/monthly_plot.png")

## 8. Rolling-origin validation

The comparison above scores every model on one test window. That is a single
number with no sense of how far it would move if the window had fallen
elsewhere.

This walks the origin forward instead: each fold trains on everything before its
test block and tests only forward. Five folds give five independent reads, and
the spread across them is the honest statement of reliability.

In [ ]:
!python src/rolling_validation.py --data data/household_daily.csv --folds 5

In [ ]:
summary = pd.read_csv("results/rolling_summary.csv")
display(summary)

folds = pd.read_csv("results/rolling_folds.csv")
display(folds.pivot(index="model", columns="fold", values="MAE").round(3))

Image("results/rolling_plot.png")

In [ ]:
# Is the gap between models bigger than the fold-to-fold noise? If the spread
# between models is small next to the standard deviation within each model, the
# ranking is not real and should not be reported as one.
ml = summary[~summary.model.str.contains("naive", case=False)]
spread = ml.mean_MAE.max() - ml.mean_MAE.min()
noise = ml.std_MAE.mean()

print(f"best-to-worst gap between models : {spread:.3f} kWh")
print(f"average fold-to-fold std dev     : {noise:.3f} kWh")
print(f"gap as a share of the noise      : {spread / noise * 100:.1f}%")
print()
if spread < 0.25 * noise:
    print("The models are statistically indistinguishable. Report the "
          "improvement over the baseline as the finding,\nnot a winning model.")
else:
    print("The gap is large enough relative to the noise to name a best model.")

## 9. Explainability (SHAP)

Gain importance ranks features but says nothing about direction or size. SHAP
values are contributions in kWh to a single day's prediction, and they sum
exactly to the difference between that prediction and the average prediction.
The script asserts that additivity holds - if it does not, the explanation is
not faithful to the model and should not be reported.

In [ ]:
!python src/explain.py --data data/household_daily.csv

In [ ]:
display(pd.read_csv("results/shap_summary.csv").head(12))
display(Markdown(open("results/shap_drivers.md").read()))

In [ ]:
Image("results/shap_beeswarm.png")

In [ ]:
# Two individual days explained end to end: the highest and lowest consumption
# days in the test window.
display(Image("results/shap_waterfall_highest_day.png"))
display(Image("results/shap_waterfall_lowest_day.png"))

## 9b. Feature-group ablation

SHAP named `other_wh_lag1` the largest single contributor. But `other_wh` is
total active energy minus the three sub-meters, so that column is a linear
combination of yesterday's total and its metered parts - all of which are also
in the feature table. SHAP divides credit between correlated features more or
less arbitrarily, so a large SHAP value is not proof that a feature carries
information nothing else carries.

Dropping a group and retraining is proof. Error barely moves: the group was
redundant. Error rises clearly: the group held signal of its own. Each group is
dropped across all five folds, and the change is compared against the full
model's own fold-to-fold variation - anything smaller than that is noise, not a
finding.

In [ ]:
%%writefile src/ablation.py
"""
Feature-group ablation: which groups of features actually earn their place.

SHAP says `other_wh_lag1` is the largest single contributor. But `other_wh` is
computed as total active energy minus the three sub-meters, so that column is a
linear combination of yesterday's total and its metered parts - all of which are
also in the feature table. SHAP splits credit between correlated features more
or less arbitrarily, so a large SHAP value does not prove a feature is carrying
information nothing else carries.

Removing a group and retraining does prove it. If the error barely moves, the
group was redundant. If it gets clearly worse, the group held signal of its own.

Each group is dropped across all five rolling-origin folds rather than one test
window, so the answer is not an artefact of where the split happened to fall.

Usage:  python ablation.py --data data/household_daily.csv
        python ablation.py --data data/household_daily.csv --folds 5

Writes results/ablation.csv, results/ablation_plot.png
"""
import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

from forecast_pipeline import (SEED, TARGET, load, make_features,
                               assert_no_leakage, mape)
from rolling_validation import folds

# Each group is a list of regex patterns matched against column names.
GROUPS = {
    "all features": [],
    "no residual lags": [r"^other_wh_"],
    "no sub-meter lags": [r"^sub[123]_wh_"],
    "no electrical lags": [r"^voltage_", r"^intensity_", r"^reactive_kvarh_"],
    "no calendar": [r"^dow", r"^month$", r"^doy", r"^is_weekend$"],
    "no rolling stats": [r"^roll_"],
    "target lags only": [r"^(?!lag_\d+$).*"],      # keep lag_1..lag_14, drop all else
}


def columns_for(group, all_cols):
    """Apply a group's drop patterns and return the surviving columns."""
    patterns = GROUPS[group]
    keep = []
    for c in all_cols:
        if any(re.match(p, c) for p in patterns):
            continue
        keep.append(c)
    return keep


def fit_xgb(Xtr, ytr):
    """Early stopping on the last 15% of the training block - still in time
    order, so nothing after the split is seen."""
    cut = int(len(Xtr) * 0.85)
    probe = XGBRegressor(n_estimators=1500, learning_rate=0.05, max_depth=4,
                         subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                         random_state=SEED, n_jobs=1,
                         early_stopping_rounds=50, eval_metric="mae")
    probe.fit(Xtr.iloc[:cut], ytr.iloc[:cut],
              eval_set=[(Xtr.iloc[cut:], ytr.iloc[cut:])], verbose=False)

    m = XGBRegressor(n_estimators=probe.best_iteration + 1, learning_rate=0.05,
                     max_depth=4, subsample=0.8, colsample_bytree=0.8,
                     reg_lambda=1.0, random_state=SEED, n_jobs=1)
    m.fit(Xtr, ytr)
    return m


def main(path, outdir, k):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)

    raw = load(path)
    feat = make_features(raw)
    assert_no_leakage(raw, feat)

    all_cols = [c for c in feat.columns if c not in ("date", TARGET)]
    n = len(feat)
    print(f"{n} rows | {len(all_cols)} features | {k} folds\n")

    rows = []
    for group in GROUPS:
        cols = columns_for(group, all_cols)
        if not cols:
            print(f"{group:<20} skipped - no features left")
            continue

        maes, mapes = [], []
        for train_end, test_end in folds(n, k):
            tr = feat.iloc[:train_end]
            te = feat.iloc[train_end:test_end]
            m = fit_xgb(tr[cols], tr[TARGET])
            p = m.predict(te[cols])
            maes.append(float(np.mean(np.abs(p - te[TARGET].to_numpy()))))
            mapes.append(mape(te[TARGET], p))

        rows.append({
            "feature_set": group,
            "n_features": len(cols),
            "dropped": len(all_cols) - len(cols),
            "mean_MAE": float(np.mean(maes)),
            "std_MAE": float(np.std(maes, ddof=1)),
            "mean_MAPE": float(np.mean(mapes)),
        })
        print(f"{group:<20} {len(cols):>3} features  "
              f"MAE {np.mean(maes):.3f} +/- {np.std(maes, ddof=1):.3f}")

    res = pd.DataFrame(rows)
    base = res.loc[res.feature_set == "all features"].iloc[0]
    res["MAE_change"] = res.mean_MAE - base.mean_MAE
    res["MAE_change_%"] = res.MAE_change / base.mean_MAE * 100

    # Is the change bigger than the fold-to-fold noise of the full model? If
    # not, the group is redundant and should not be called a key driver.
    res["beyond_noise"] = res.MAE_change.abs() > base.std_MAE
    res = res.round(3)

    print("\n" + "=" * 86)
    print(f"ABLATION  (XGBoost, mean of {k} rolling-origin folds, "
          f"full-model fold-to-fold std = {base.std_MAE:.3f})")
    print("=" * 86)
    print(res.to_string(index=False))

    print("\nReading it:")
    for _, r in res.iterrows():
        if r.feature_set == "all features":
            continue
        if not r.beyond_noise:
            print(f"  {r.feature_set:<20} redundant - error moved "
                  f"{r.MAE_change:+.3f} kWh, inside the noise")
        elif r.MAE_change > 0:
            print(f"  {r.feature_set:<20} CARRIES SIGNAL - error rose "
                  f"{r.MAE_change:+.3f} kWh ({r['MAE_change_%']:+.1f}%)")
        else:
            print(f"  {r.feature_set:<20} HURT the model - error FELL "
                  f"{r.MAE_change:+.3f} kWh when removed")

    res.to_csv(outdir / "ablation.csv", index=False)

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        plot = res[res.feature_set != "all features"].sort_values("MAE_change")
        colours = ["#888888" if not b else "#c0392b"
                   for b in plot.beyond_noise]
        fig, ax = plt.subplots(figsize=(7, 3.6))
        ax.barh(plot.feature_set, plot.MAE_change, color=colours)
        ax.axvline(0, color="black", lw=0.8)
        ax.axvline(base.std_MAE, color="black", ls=":", lw=0.8,
                   label="fold-to-fold noise")
        ax.axvline(-base.std_MAE, color="black", ls=":", lw=0.8)
        ax.set_xlabel("change in MAE when the group is removed (kWh)")
        ax.set_title("Feature-group ablation")
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(outdir / "ablation_plot.png", dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"\n(plot skipped: {e})")

    print(f"\nwrote -> {outdir}/ablation.csv, ablation_plot.png")
    return res


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--outdir", default="results")
    ap.add_argument("--folds", type=int, default=5)
    a = ap.parse_args()
    main(a.data, a.outdir, a.folds)

In [ ]:
!python src/ablation.py --data data/household_daily.csv --folds 5

In [ ]:
display(pd.read_csv("results/ablation.csv"))
Image("results/ablation_plot.png")

## 9c. Train the deliverable models, then forecast

Everything above measures models on history that already happened. This section
produces what a client actually uses: fitted models saved to disk, and forecasts
for days that have not occurred.

Two models are saved, because one feature set cannot do both jobs.

`next_day` keeps every feature, including lags of the end-of-day columns -
voltage, the sub-meters, the residual. Yesterday's values for those *are* known
when forecasting tomorrow, so this is the more accurate model. But only one
step ahead.

`horizon` drops all of them and keeps target lags, rolling statistics and
calendar only. Forecasting the day after tomorrow would need tomorrow's
sub-meter readings, which will never exist. With nothing end-of-day in its
inputs, this model can be run recursively: each prediction becomes the next
step's lag.

The ablation in 9b already measured what that costs - the reduced feature set
gives up a few percent of accuracy. Better to know the price than to discover
the limitation when the client asks for a month.

In [ ]:
%%writefile src/train_model.py
"""
Train the final models on all available history and save them to disk.

Everything else in this project measures models on held-out history. This step
produces the artefact a forecast actually comes from: a fitted model, the exact
feature list it expects, and a record of what it was trained on.

Two models are saved, because the feature set does not support both jobs.

  next_day   Every feature, including lags of the end-of-day columns
             (voltage, sub-meters, residual). Yesterday's values for those ARE
             known when forecasting tomorrow, so this model is the most
             accurate - but only one day ahead.

  horizon    Target lags, rolling statistics and calendar only. Nothing derived
             from the end-of-day columns, because forecasting day+2 would need
             day+1's sub-meter readings, which will not exist. This model can
             therefore be run recursively for any number of days.

The ablation showed the reduced feature set costs about 8% MAE, which is the
price of forecasting further than one day out. Worth knowing rather than
discovering it when the client asks for a month.

Usage:  python train_model.py --data data/household_daily.csv

Writes models/next_day.joblib, models/horizon.joblib, models/model_card.md
"""
import argparse
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

from forecast_pipeline import (SEED, TARGET, SAME_DAY_ONLY, load,
                               make_features, assert_no_leakage, mape)

PARAMS = dict(learning_rate=0.05, max_depth=4, subsample=0.8,
              colsample_bytree=0.8, reg_lambda=1.0,
              random_state=SEED, n_jobs=1)


def feature_sets(feat):
    """Split the columns into the two usable sets."""
    cols = [c for c in feat.columns if c not in ("date", TARGET)]

    # Anything derived from a column only known at the end of a day.
    derived = tuple(f"{c}_" for c in SAME_DAY_ONLY)
    horizon = [c for c in cols if not c.startswith(derived)]

    return {"next_day": cols, "horizon": horizon}


def tune_then_refit(feat, cols, val_frac=0.15):
    """Choose n_estimators on a chronological holdout, then refit on everything.

    The holdout exists only to pick the tree count. Once that number is fixed
    the model is retrained on all available history, because a forecast for
    tomorrow should use every day up to today - not stop short to preserve a
    test set that has already served its purpose.
    """
    X, y = feat[cols], feat[TARGET]
    cut = int(len(feat) * (1 - val_frac))

    probe = XGBRegressor(n_estimators=2000, early_stopping_rounds=50,
                         eval_metric="mae", **PARAMS)
    probe.fit(X.iloc[:cut], y.iloc[:cut],
              eval_set=[(X.iloc[cut:], y.iloc[cut:])], verbose=False)
    best_n = probe.best_iteration + 1

    # Holdout score at that tree count - the honest expectation for new days.
    p = probe.predict(X.iloc[cut:])
    holdout = {
        "val_days": int(len(feat) - cut),
        "val_from": str(feat.date.iloc[cut].date()),
        "val_MAE": float(np.mean(np.abs(p - y.iloc[cut:].to_numpy()))),
        "val_MAPE": float(mape(y.iloc[cut:], p)),
    }

    final = XGBRegressor(n_estimators=best_n, **PARAMS)
    final.fit(X, y)
    return final, best_n, holdout


def main(path, outdir):
    import joblib

    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)

    raw = load(path)
    feat = make_features(raw)
    assert_no_leakage(raw, feat)

    sets = feature_sets(feat)
    digest = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    cards = {}

    for name, cols in sets.items():
        model, best_n, holdout = tune_then_refit(feat, cols)

        meta = {
            "name": name,
            "features": cols,
            "n_features": len(cols),
            "n_estimators": int(best_n),
            "target": TARGET,
            "trained_rows": int(len(feat)),
            "trained_from": str(feat.date.min().date()),
            "trained_to": str(feat.date.max().date()),
            "data_file": Path(path).name,
            "data_sha256_12": digest[:12],
            "seed": SEED,
            "params": PARAMS,
            "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            **holdout,
        }

        joblib.dump({"model": model, "meta": meta}, outdir / f"{name}.joblib")
        cards[name] = meta

        print(f"{name:<9} {len(cols):>2} features, {best_n:>3} trees | "
              f"holdout MAE {holdout['val_MAE']:.3f} "
              f"MAPE {holdout['val_MAPE']:.1f}% "
              f"on {holdout['val_days']} days from {holdout['val_from']}")

    nd, hz = cards["next_day"], cards["horizon"]
    cost = (hz["val_MAE"] - nd["val_MAE"]) / nd["val_MAE"] * 100

    (outdir / "model_card.md").write_text(f"""# Model card

Household daily electricity forecasting, Forecasting 5 (ICT30016).
Client: ARC Research Hub for Integrated Energy Storage Solutions.

Trained {nd['trained_at']} on `{nd['data_file']}`
(SHA-256 `{nd['data_sha256_12']}...`), {nd['trained_rows']} days from
{nd['trained_from']} to {nd['trained_to']}. Seed {SEED}.

## The two models

| | `next_day` | `horizon` |
|---|---|---|
| Features | {nd['n_features']} | {hz['n_features']} |
| Trees | {nd['n_estimators']} | {hz['n_estimators']} |
| Holdout MAE (kWh) | {nd['val_MAE']:.3f} | {hz['val_MAE']:.3f} |
| Holdout MAPE | {nd['val_MAPE']:.1f}% | {hz['val_MAPE']:.1f}% |
| Forecast range | 1 day | any |

`next_day` uses lags of the end-of-day columns (voltage, current, reactive
energy, the three sub-meters and the residual). Yesterday's values for those
are known when forecasting tomorrow, so it is the more accurate model - but
only one step ahead.

`horizon` uses target lags, rolling statistics and calendar features only, so
it can be run recursively: each prediction becomes the next step's lag. It
costs {cost:+.1f}% MAE against `next_day`, which is the measured price of
forecasting further than one day out.

## Intended use

Short-horizon forecasts of total daily active energy for one household, to
inform storage sizing and dispatch. Daily predictions aggregate to a monthly
view with lower percentage error than the daily figures.

## Limits

- **One household.** Trained on a single house in Sceaux, France
  (UCI, Hebrail & Berard 2012). Nothing here establishes transfer to other
  households, other climates or Australian consumption patterns.
- **No weather input.** Temperature drives household consumption and is not in
  this dataset. Lagged consumption absorbs some of it indirectly.
- **Recursive error growth.** `horizon` feeds its own predictions back in, so
  accuracy decays with distance. The holdout figure above is a one-step score
  and is optimistic for longer horizons.
- **Retraining.** Both models are fixed at the date above. Consumption habits
  drift, so they should be retrained when new data arrives rather than trusted
  indefinitely.

## Validation performed

Chronological splits throughout; no random k-fold (Bergmeir & Benitez 2012).
Leakage assertions run on every build. Rolling-origin validation over five
folds, a feature-group ablation, and TreeSHAP attribution with an additivity
check. See `results/` and `tests/`.
""")

    print(f"\nhorizon model costs {cost:+.1f}% MAE vs next_day - the price of "
          f"forecasting past tomorrow")
    print(f"\nwrote -> {outdir}/next_day.joblib, horizon.joblib, model_card.md")
    return cards


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--outdir", default="models")
    a = ap.parse_args()
    main(a.data, a.outdir)

In [ ]:
!pip install -q joblib
!python src/train_model.py --data data/household_daily.csv

In [ ]:
display(Markdown(open("models/model_card.md").read()))

In [ ]:
%%writefile src/predict.py
"""
Produce actual forecasts for days that have not happened yet.

This is the part a client uses. Everything else in the project measures the
models; this runs them forward.

Two things happen:

  Tomorrow      One step ahead with the `next_day` model, which can use
                yesterday's end-of-day readings because they exist.

  The next N    Recursive forecasting with the `horizon` model. Each predicted
                day becomes the next day's lag. Error compounds as it goes, so
                the output is honest about that rather than presenting day 30
                with the same confidence as day 1.

Feature rows are built from the TAIL of the series by position, which is how
make_features builds them during training - lag_1 means "the previous row", not
"the previous calendar day". At the end of the series those are the same thing,
because the dropped partial days sit in the interior.

Usage:  python predict.py --data data/household_daily.csv
        python predict.py --data data/household_daily.csv --days 30

Writes results/forecast.csv, results/forecast_monthly.csv, results/forecast.png
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from forecast_pipeline import TARGET, load

LAGS = (1, 2, 3, 7, 14)
WINDOWS = (7, 14, 28)


def calendar_features(date):
    d = pd.Timestamp(date)
    return {
        "dow": d.dayofweek,
        "month": d.month,
        "doy": d.dayofyear,
        "is_weekend": int(d.dayofweek >= 5),
        "dow_sin": np.sin(2 * np.pi * d.dayofweek / 7),
        "dow_cos": np.cos(2 * np.pi * d.dayofweek / 7),
        "doy_sin": np.sin(2 * np.pi * d.dayofyear / 365.25),
        "doy_cos": np.cos(2 * np.pi * d.dayofyear / 365.25),
    }


def history_features(series, date):
    """Target lags and rolling statistics for `date`, from the values that
    precede it. `series` is a plain list of past kWh values in time order."""
    row = {}
    for lag in LAGS:
        row[f"lag_{lag}"] = series[-lag]
    for w in WINDOWS:
        window = series[-w:]
        row[f"roll_mean_{w}"] = float(np.mean(window))
        row[f"roll_std_{w}"] = float(np.std(window, ddof=1))
    row.update(calendar_features(date))
    return row


def same_day_lag_features(daily, date):
    """Lags of the end-of-day columns, for a one-step-ahead forecast only.

    For `date` = the day after the last observed day, lag1 is the last observed
    row and lag7 is seven rows back - both real measurements.
    """
    row = {}
    for c in ("reactive_kvarh", "voltage", "intensity",
              "sub1_wh", "sub2_wh", "sub3_wh", "other_wh"):
        if c not in daily.columns:
            continue
        row[f"{c}_lag1"] = float(daily[c].iloc[-1])
        row[f"{c}_lag7"] = float(daily[c].iloc[-7])
    return row


def load_model(path):
    import joblib
    bundle = joblib.load(path)
    return bundle["model"], bundle["meta"]


def forecast(daily, days, models_dir):
    hist = daily[TARGET].tolist()
    last = pd.Timestamp(daily["date"].iloc[-1])

    nd_model, nd_meta = load_model(Path(models_dir) / "next_day.joblib")
    hz_model, hz_meta = load_model(Path(models_dir) / "horizon.joblib")

    rows = []

    # --- step 1: tomorrow, with the more accurate model -------------------
    d1 = last + pd.Timedelta(days=1)
    row = history_features(hist, d1)
    row.update(same_day_lag_features(daily, d1))
    X = pd.DataFrame([row])[nd_meta["features"]]
    p1 = float(nd_model.predict(X)[0])

    rows.append({"date": d1, "step": 1, "predicted_kwh": p1,
                 "model": "next_day", "expected_MAE": nd_meta["val_MAE"]})
    hist.append(p1)

    # --- steps 2..N: recursive, reduced feature set ------------------------
    # Error compounds, so the quoted uncertainty is widened with the square
    # root of the step count. That is the standard assumption for errors that
    # accumulate roughly independently - it is an estimate, not a measurement,
    # and is labelled as such in the output.
    for step in range(2, days + 1):
        d = last + pd.Timedelta(days=step)
        X = pd.DataFrame([history_features(hist, d)])[hz_meta["features"]]
        p = float(hz_model.predict(X)[0])

        rows.append({"date": d, "step": step, "predicted_kwh": p,
                     "model": "horizon",
                     "expected_MAE": hz_meta["val_MAE"] * np.sqrt(step)})
        hist.append(p)

    out = pd.DataFrame(rows)
    out["lower_kwh"] = (out.predicted_kwh - out.expected_MAE).clip(lower=0)
    out["upper_kwh"] = out.predicted_kwh + out.expected_MAE
    return out, nd_meta, hz_meta


def to_monthly(fc, last_observed):
    """Monthly totals from the daily forecasts, flagging any month the forecast
    only partly covers - a stub month is not a monthly total."""
    f = fc.copy()
    f["month"] = f.date.dt.to_period("M")

    g = (f.groupby("month")
          .agg(days_forecast=("predicted_kwh", "size"),
               predicted_kwh=("predicted_kwh", "sum"),
               lower_kwh=("lower_kwh", "sum"),
               upper_kwh=("upper_kwh", "sum"))
          .reset_index())

    g["days_in_month"] = g.month.dt.days_in_month
    g["complete"] = g.days_forecast == g.days_in_month
    g["month"] = g.month.astype(str)
    return g


def main(path, models_dir, outdir, days):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)

    daily = load(path)
    last = pd.Timestamp(daily["date"].iloc[-1])
    print(f"history: {len(daily)} days to {last.date()}")

    fc, nd_meta, hz_meta = forecast(daily, days, models_dir)

    print(f"\nTOMORROW ({fc.date.iloc[0].date()}): "
          f"{fc.predicted_kwh.iloc[0]:.2f} kWh "
          f"(+/- {fc.expected_MAE.iloc[0]:.2f} expected error)")

    print(f"\nnext {days} days")
    show = fc.head(10)[["date", "step", "predicted_kwh",
                        "lower_kwh", "upper_kwh", "model"]].copy()
    show["date"] = show.date.dt.date
    print(show.round(2).to_string(index=False))
    if days > 10:
        print(f"  ... {days - 10} more rows in results/forecast.csv")

    total = fc.predicted_kwh.sum()
    print(f"\ntotal over {days} days: {total:.1f} kWh "
          f"({total / days:.2f} kWh/day average)")

    monthly = to_monthly(fc, last)
    print("\nmonthly view")
    print(monthly.round(1).to_string(index=False))
    if not monthly.complete.all():
        part = monthly.loc[~monthly.complete, "month"].tolist()
        print(f"  partial months (forecast does not cover the whole month): "
              f"{', '.join(part)}")

    fc_out = fc.copy()
    fc_out["date"] = fc_out.date.dt.date
    fc_out.round(4).to_csv(outdir / "forecast.csv", index=False)
    monthly.round(4).to_csv(outdir / "forecast_monthly.csv", index=False)

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        tail = daily.tail(60)
        fig, ax = plt.subplots(figsize=(12, 3.6))
        ax.plot(tail.date, tail[TARGET], lw=0.9, color="#333333",
                label="observed")
        ax.plot(fc.date, fc.predicted_kwh, lw=1.4, color="#c0392b",
                label="forecast")
        ax.fill_between(fc.date, fc.lower_kwh, fc.upper_kwh,
                        color="#c0392b", alpha=0.15,
                        label="expected error band")
        ax.axvline(last, color="#999999", ls=":", lw=1)
        ax.set_ylabel("kWh/day")
        ax.set_title(f"Last 60 observed days and the next {days} forecast")
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(outdir / "forecast.png", dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"\n(plot skipped: {e})")

    print(f"\nwrote -> {outdir}/forecast.csv, forecast_monthly.csv, forecast.png")
    print("\nNote: the band widens with the square root of the step count. "
          "It is an\nestimate of accumulating error, not a measured prediction "
          "interval.")
    return fc, monthly


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--models", default="models")
    ap.add_argument("--outdir", default="results")
    ap.add_argument("--days", type=int, default=14)
    a = ap.parse_args()
    main(a.data, a.models, a.outdir, a.days)

In [ ]:
!python src/predict.py --data data/household_daily.csv --days 30

In [ ]:
display(pd.read_csv("results/forecast.csv").head(10))
display(pd.read_csv("results/forecast_monthly.csv"))
Image("results/forecast.png")

## 10. Full test run

Now that the daily table and the manifest exist, the two regression tests that
skipped in step 4 can run. This is the evidence for the QA plan: the cleaning
rules, the conversion and the leakage guards are all checked, not asserted.

In [ ]:
!python -m pytest tests/ -v

## 11. Download everything

One zip rather than fifteen separate downloads. Contains the daily table, the
manifest and data dictionary, every result CSV, every chart, and the project
source files.

In [ ]:
# Everything except raw/ - the raw file is 127 MB and is not a project output.
with zipfile.ZipFile("forecasting5_outputs.zip", "w",
                     zipfile.ZIP_DEFLATED) as z:
    for folder in ("data", "results", "models", "src", "tests"):
        for root, _, names in os.walk(folder):
            for n in names:
                if n.endswith(".pyc") or "__pycache__" in root:
                    continue
                p = os.path.join(root, n)
                z.write(p, p)

size = os.path.getsize("forecasting5_outputs.zip") / 1e6
print(f"forecasting5_outputs.zip  ({size:.1f} MB)")
with zipfile.ZipFile("forecasting5_outputs.zip") as z:
    for n in sorted(z.namelist()):
        print("  ", n)

In [ ]:
files.download("forecasting5_outputs.zip")

## What is outstanding

- **LSTM / GRU.** Sequence models need the data as sliding windows rather than
  lag columns, so they sit in a separate notebook. Their job is to confirm the
  tree-based choice, not to be the deliverable.
- **Generalisation.** Everything here describes one house in Sceaux, France.
  Nothing establishes that the approach transfers to other households, other
  climates or Australian consumption patterns. A second dataset would be needed
  to claim that.
- **Weather.** Temperature is a known driver of household consumption and is not
  in this dataset. Lagged consumption absorbs some of it indirectly.